# 01 — Build an SFT dataset for scientific mechanisms

**Learning goal.** Turn openly licensed scientific source material into
self-contained *how/why* tasks. A teacher model drafts each task; a separate
critic call checks it against the source. The student never sees the source.

```
open scientific source
        │  teacher + critic (gpt-5.6-terra)
        ▼
self-contained mechanism task
        │
        ├── visible causal explanation
        ├── evidence quoted from the task
        └── concise mechanistic answer
```

This course deliberately excludes calculation and numerical prediction tasks.
The target is a causal chain: **condition → intermediate process → outcome**.

## Before you run

From the repository root:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[dev]"
export OPENAI_API_KEY="..."
hf auth login
jupyter lab
```

The generation is real, paid API use—there is no offline substitute. Each source
normally uses two calls (teacher and critic). Results are appended incrementally,
so interrupting and rerunning is safe.

## Configuration

Every editable setting is defined below. The OpenAI key is the only required
environment variable. Hugging Face uses `hf auth login` by default; the commented
`HF_TOKEN` line is an optional alternative and should never contain a pasted token.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import JSON, Markdown, display

from science_course.data import (
    build_sft_dataset,
    read_jsonl,
    stream_open_science_sources,
    write_jsonl,
)
from science_course.hub import require_hf_namespace
from science_course.teacher import (
    DEFAULT_TEACHER_MODEL,
    generate_canonical_tasks,
    require_openai_key,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RAW_SOURCES = DATA / "raw" / "open_science_sources.jsonl"
ACCEPTED = DATA / "canonical" / "mechanism_tasks.jsonl"
REJECTED = DATA / "canonical" / "rejected_tasks.jsonl"
SFT_DISK = DATA / "processed" / "sft"

# Dataset authoring
TEACHER_MODEL = DEFAULT_TEACHER_MODEL
SOURCE_DATASET_ID = "common-pile/peS2o"
SOURCE_SPLIT = "train"
MAX_SOURCE_PAPERS = 120
MAX_SOURCE_RECORDS_SCANNED = 20_000
MIN_SOURCE_CHARS = 1_200
MAX_SOURCE_CHARS = 6_000
RANDOM_SEED = 17

# Hugging Face publication
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
PUSH_DATASETS_TO_HUB = True
DATASET_PRIVATE = False
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

require_openai_key()
assert TEACHER_MODEL == "gpt-5.6-terra", (
    "This class notebook is tested with the requested teacher model: "
    "gpt-5.6-terra"
)
if PUSH_DATASETS_TO_HUB:
    require_hf_namespace(DATASET_HF_REPO, token=HF_TOKEN)

sns.set_theme(style="whitegrid", context="talk")
print(
    {
        "root": str(ROOT),
        "teacher_model": TEACHER_MODEL,
        "sources": MAX_SOURCE_PAPERS,
        "dataset_hub_repo": DATASET_HF_REPO,
    }
)

## 1. Acquire open source material

We stream `common-pile/peS2o`, derived from openly licensed scientific papers.
Per-document license metadata is filtered before any teacher call. Only a bounded
source excerpt is sent to the teacher; the record retains its paper ID, URL,
license, split, and content hash for provenance.

This source text is **authoring material**, not model input after fine-tuning.

In [ ]:
cached_sources = read_jsonl(RAW_SOURCES)
if len(cached_sources) == MAX_SOURCE_PAPERS:
    sources = cached_sources
    print(f"Reusing {len(sources)} source records from {RAW_SOURCES}")
else:
    sources = stream_open_science_sources(
        dataset_id=SOURCE_DATASET_ID,
        split=SOURCE_SPLIT,
        max_papers=MAX_SOURCE_PAPERS,
        max_scanned=MAX_SOURCE_RECORDS_SCANNED,
        min_chars=MIN_SOURCE_CHARS,
        max_chars=MAX_SOURCE_CHARS,
        seed=RANDOM_SEED,
    )
    write_jsonl(RAW_SOURCES, sources)
    print(
        f"Refreshed the source cache with {len(sources)} records at {RAW_SOURCES}"
    )

if not sources:
    raise RuntimeError(
        "No open sources passed the filters. Increase MAX_SOURCE_PAPERS and "
        "rerun acquisition."
    )

source_frame = pd.DataFrame(sources)
display(
    source_frame[
        ["paper_id", "title", "source_license", "split", "source_url"]
    ].head()
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
source_frame["source_license"].value_counts().plot.bar(
    ax=axes[0], color="#315c8c", title="Open-license provenance"
)
source_frame["split"].value_counts().reindex(
    ["sft_train", "sft_validation", "grpo_train", "grpo_validation", "test"]
).plot.bar(ax=axes[1], color="#d97732", title="Paper-level split")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 2. Author and critique mechanism tasks

The teacher produces a structured record. It must:

- ask a qualitative **how/why** question;
- include every observation needed to answer;
- provide an ordered causal rubric and explicit cause→effect links;
- quote evidence from the newly written task itself; and
- avoid arithmetic, numerical targets, and unsupported claims.

A second call sees both the source and draft and can reject it. Rejections are
useful audit data, not silently discarded failures.

In [ ]:
canonical_all = generate_canonical_tasks(
    sources,
    accepted_path=ACCEPTED,
    rejected_path=REJECTED,
    model=TEACHER_MODEL,
)
active_paper_ids = {row["paper_id"] for row in sources}
canonical = [
    row for row in canonical_all if row["paper_id"] in active_paper_ids
]
rejected = [
    row
    for row in read_jsonl(REJECTED)
    if row.get("paper_id") in active_paper_ids
]
print(
    {
        "accepted": len(canonical),
        "rejected": len(rejected),
        "acceptance_rate": len(canonical) / max(len(canonical) + len(rejected), 1),
    }
)

if not canonical:
    raise RuntimeError("No tasks were accepted. Inspect the rejection audit file.")

In [ ]:
example = canonical[0]
display(Markdown("### Student-visible task"))
display(Markdown(example["task"]))
display(Markdown("### Reference response"))
display(
    Markdown(
        f"**Reasoning:** {example['reasoning']}\n\n"
        f"**Evidence:** “{example['evidence']}”\n\n"
        f"**Answer:** {example['answer']}"
    )
)
display(Markdown("### Hidden causal rubric"))
display(
    JSON(
        {
            "mechanism_steps": example["mechanism_steps"],
            "causal_links": example["causal_links"],
            "required_concepts": example["required_concepts"],
        }
    )
)

## 3. Audit causal structure

This plot does not claim that a longer explanation is better. It checks whether
the dataset contains explicit multi-step causal supervision rather than isolated
answer labels.

In [ ]:
audit = pd.DataFrame(
    {
        "split": [row["split"] for row in canonical],
        "task_chars": [len(row["task"]) for row in canonical],
        "mechanism_steps": [len(row["mechanism_steps"]) for row in canonical],
        "causal_links": [len(row["causal_links"]) for row in canonical],
    }
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(audit, x="mechanism_steps", discrete=True, ax=axes[0], color="#315c8c")
axes[0].set_title("Explicit steps per causal rubric")
sns.scatterplot(
    audit,
    x="task_chars",
    y="causal_links",
    hue="split",
    ax=axes[1],
    s=80,
)
axes[1].set_title("Task size versus cause→effect links")
plt.tight_layout()
plt.show()

## 4. Project the SFT view

SFT receives a conversational `(prompt, completion)` pair. The completion is the
tagged reference response. Raw source text and the hidden rubric are intentionally
absent. Paper IDs, URL, and license remain for traceability.

In [ ]:
import shutil

sft = build_sft_dataset(canonical)
if len(sft["train"]) == 0 or len(sft["validation"]) == 0:
    raise RuntimeError(
        "The paper-level split produced an empty SFT partition. Increase "
        "MAX_SOURCE_PAPERS, then rerun generation."
    )
if SFT_DISK.exists():
    shutil.rmtree(SFT_DISK)
sft.save_to_disk(SFT_DISK)
for split_name, split_data in sft.items():
    split_data.to_parquet(DATA / "processed" / f"sft_{split_name}.parquet")
if PUSH_DATASETS_TO_HUB:
    sft.push_to_hub(
        DATASET_HF_REPO,
        config_name="sft",
        private=DATASET_PRIVATE,
        token=HF_TOKEN,
        commit_message="Publish mechanism SFT splits",
    )

assert "source_text" not in sft["train"].column_names
assert "mechanism_steps" not in sft["train"].column_names
print(sft)
display(JSON(sft["train"][0]))

## Result

You now have:

- an auditable canonical dataset with source provenance and hidden causal rubrics;
- an explicit rejection log;
- an SFT dataset whose model input is only a self-contained mechanism task.

The processed SFT splits are also published under the `sft` configuration of
`lamm-mit/scientific-sft-grpo-data`. The private canonical source-text audit file
remains local and is never uploaded.

Continue with **02_build_mechanism_grpo_dataset.ipynb**.